In [2]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.storage.database import get_engine

plt.style.use('seaborn-v0_8-whitegrid')
engine = get_engine()

with engine.connect() as conn:
    prices = pd.read_sql("""
        SELECT p.date, p.close, p.adjusted_close, p.volume,
               p.week52_low, p.week52_high, s.ticker, s.sector
        FROM prices p
        JOIN securities s ON p.security_id = s.security_id
        WHERE s.security_type = 'equity'
        AND p.close IS NOT NULL
        AND p.week52_low IS NOT NULL
        AND p.week52_high IS NOT NULL
    """, conn)

    returns = pd.read_sql("""
        SELECT r.date, r.adj_daily_return, s.ticker
        FROM returns r
        JOIN securities s ON r.security_id = s.security_id
        WHERE s.security_type = 'equity'
        AND r.adj_daily_return IS NOT NULL
        AND ABS(r.adj_daily_return) <= 1.0
    """, conn)

prices['date'] = pd.to_datetime(prices['date'])
returns['date'] = pd.to_datetime(returns['date'])

print(f"Prices: {len(prices):,} rows with 52-week range data")
print(f"Returns: {len(returns):,} rows")

Prices: 280,718 rows with 52-week range data
Returns: 249,488 rows


In [2]:
# Factor 1: 52-week low proximity
# How close is current price to 52-week low?
# Formula: (price - 52w_low) / (52w_high - 52w_low)
# 0 = at 52-week low (cheapest) 
# 1 = at 52-week high (most expensive)
# Low values = value signal (stock is cheap relative to recent range)

prices['range'] = prices['week52_high'] - prices['week52_low']
prices['proximity'] = np.where(
    prices['range'] > 0,
    (prices['close'] - prices['week52_low']) / prices['range'],
    np.nan
)

# Sanity check
valid = prices['proximity'].dropna()
print(f"Valid proximity values: {len(valid):,}")
print(f"Range: {valid.min():.3f} to {valid.max():.3f}")
print(f"Mean:  {valid.mean():.3f}")
print(f"Values outside [0,1]: {((valid < 0) | (valid > 1)).sum()}")
print(f"\nSample:")
print(prices[['ticker','date','close','week52_low','week52_high','proximity']]
      .dropna(subset=['proximity']).head(8).to_string(index=False))

Valid proximity values: 262,571
Range: -26.429 to 507.857
Mean:  2.717
Values outside [0,1]: 115233

Sample:
ticker       date  close  week52_low  week52_high  proximity
  EGAD 2007-01-02  52.00        22.0        57.00   0.857143
  KAPC 2007-01-02 100.00       111.0       148.00  -0.297297
  KUKZ 2007-01-02  43.50        67.5        89.00  -1.116279
   REA 2007-01-02  25.50        14.5        23.50   1.222222
  SASN 2007-01-02 140.00        10.5        13.60  41.774194
   WTK 2007-01-02 120.00       180.0       290.00  -0.545455
   C&G 2007-01-02  50.00        21.0        29.00   3.625000
  FIRE 2007-01-02  24.75         3.4         5.95   8.372549


In [3]:
# Factor: 6-month price momentum (long horizon)
# Stocks with worst 6-month returns = value-like signal
# 126 trading days ≈ 6 months

print("Computing 6-month and 12-month momentum factors...")

long_mom = []

for ticker, group in prices.groupby('ticker'):
    g = group.sort_values('date').copy()
    adj = pd.to_numeric(g['adjusted_close'], errors='coerce')
    log_prices = np.log(adj)
    
    # 6-month momentum (skip 1 month = 21 days)
    mom_6m = log_prices.shift(21) - log_prices.shift(126 + 21)
    
    # 12-month momentum (skip 1 month)
    mom_12m = log_prices.shift(21) - log_prices.shift(252 + 21)
    
    long_mom.append(pd.DataFrame({
        'ticker': ticker,
        'date': g['date'].values,
        'mom_6m': mom_6m.values,
        'mom_12m': mom_12m.values,
    }))

long_mom_df = pd.concat(long_mom, ignore_index=True)

valid_6m = long_mom_df['mom_6m'].notna().sum()
valid_12m = long_mom_df['mom_12m'].notna().sum()
print(f"Valid 6m observations:  {valid_6m:,}")
print(f"Valid 12m observations: {valid_12m:,}")
print(f"\nSample:")
print(long_mom_df.dropna(subset=['mom_6m']).head(5).to_string(index=False))

Computing 6-month and 12-month momentum factors...
Valid 6m observations:  268,387
Valid 12m observations: 257,994

Sample:
ticker       date   mom_6m  mom_12m
  ABSA 2013-08-02 0.018928      NaN
  ABSA 2013-08-05 0.015699      NaN
  ABSA 2013-08-06 0.049546      NaN
  ABSA 2013-08-07 0.070690      NaN
  ABSA 2013-08-08 0.079553      NaN


In [4]:
# Quintile analysis — long horizon momentum
merged = long_mom_df.merge(
    returns[['ticker','date','adj_daily_return']],
    on=['ticker','date']
).dropna(subset=['mom_6m'])

merged = merged.sort_values(['ticker','date'])
merged['fwd_return'] = merged.groupby('ticker')['adj_daily_return'].shift(-1)
merged = merged.dropna(subset=['fwd_return'])

print("=== LONG-HORIZON MOMENTUM QUINTILES ===\n")
print(f"{'Factor':<10} {'Q1 (losers)':<14} {'Q5 (winners)':<14} {'L/S Spread':<12} {'Q1 t-stat'}")
print("-" * 60)

for col in ['mom_6m', 'mom_12m']:
    data = merged.dropna(subset=[col])
    
    quintile_rets = {q: [] for q in [1, 5]}
    
    for date, group in data.groupby('date'):
        if len(group) < 15:
            continue
        try:
            group = group.copy()
            group['q'] = pd.qcut(
                group[col], q=5, labels=[1,2,3,4,5], duplicates='drop'
            )
            for q in [1, 5]:
                q_ret = group[group['q']==q]['fwd_return']
                if len(q_ret) >= 3:
                    quintile_rets[q].append(q_ret.mean())
        except Exception:
            continue
    
    q1 = pd.Series(quintile_rets[1]).dropna()
    q5 = pd.Series(quintile_rets[5]).dropna()
    
    q1_ann = q1.mean() * 252 * 100
    q5_ann = q5.mean() * 252 * 100
    spread = q5_ann - q1_ann
    t1 = q1.mean() / (q1.std() / np.sqrt(len(q1)))
    
    print(f"{col:<10} {q1_ann:>+10.1f}%   {q5_ann:>+10.1f}%   "
          f"{spread:>+8.1f}%   {t1:>+6.2f}")

print("\nComparison with short-horizon (20d) factor:")
print(f"{'mom_20d':<10} {'  +83.2%':<14} {'  -71.3%':<14} {' -154.5%':<12} {' +5.51'}")

=== LONG-HORIZON MOMENTUM QUINTILES ===

Factor     Q1 (losers)    Q5 (winners)   L/S Spread   Q1 t-stat
------------------------------------------------------------
mom_6m           -7.9%         -0.0%       +7.8%    -0.53
mom_12m         -12.5%         +8.4%      +20.9%    -0.83

Comparison with short-horizon (20d) factor:
mom_20d      +83.2%         -71.3%        -154.5%      +5.51


In [5]:
print("""
=== FINDING — LONG HORIZON VALUE PROXIES ===

6-month momentum:  Q1 return -7.9%,  t-stat -0.53  NOT significant
12-month momentum: Q1 return -12.5%, t-stat -0.83  NOT significant

CONCLUSION:
Long-horizon losers do NOT outperform in NSE.
They continue underperforming — genuine fundamental deterioration.

The mean reversion effect is EXCLUSIVELY short-horizon (5-20 days).
Beyond 20 days, the signal reverses.

This eliminates long-horizon value proxies as factors.
True value factors require fundamental data:
  - Price-to-book ratio
  - Dividend yield  
  - Earnings yield

These require ingesting the individual stock files.
Added to research backlog.

CURRENT FACTOR RANKING:
  1. 20d mean reversion — CONFIRMED, t-stat 5.51
  2. 6m/12m value proxies — REJECTED, not significant
""")


=== FINDING — LONG HORIZON VALUE PROXIES ===

6-month momentum:  Q1 return -7.9%,  t-stat -0.53  NOT significant
12-month momentum: Q1 return -12.5%, t-stat -0.83  NOT significant

CONCLUSION:
Long-horizon losers do NOT outperform in NSE.
They continue underperforming — genuine fundamental deterioration.

The mean reversion effect is EXCLUSIVELY short-horizon (5-20 days).
Beyond 20 days, the signal reverses.

This eliminates long-horizon value proxies as factors.
True value factors require fundamental data:
  - Price-to-book ratio
  - Dividend yield  
  - Earnings yield

These require ingesting the individual stock files.
Added to research backlog.

CURRENT FACTOR RANKING:
  1. 20d mean reversion — CONFIRMED, t-stat 5.51
  2. 6m/12m value proxies — REJECTED, not significant



In [6]:
# Factor: Size
# Proxy: average daily volume over past 60 days
# Low volume = small cap proxy
# High volume = large cap proxy
# Hypothesis: low volume (small) stocks outperform high volume (large) stocks

print("Computing size factor...")

size_rows = []

for ticker, group in prices.groupby('ticker'):
    g = group.sort_values('date').copy()
    vol = pd.to_numeric(g['volume'], errors='coerce')
    
    # Rolling 60-day average volume as size proxy
    rolling_vol = vol.rolling(60, min_periods=30).mean()
    
    size_rows.append(pd.DataFrame({
        'ticker': ticker,
        'date': g['date'].values,
        'size_proxy': rolling_vol.values,
    }))

size_df = pd.concat(size_rows, ignore_index=True)
size_df = size_df.dropna(subset=['size_proxy'])
size_df = size_df[size_df['size_proxy'] > 0]

print(f"Valid size observations: {len(size_df):,}")
print(f"\nLargest stocks (high volume):")
largest = size_df.groupby('ticker')['size_proxy'].mean().nlargest(10)
print(largest.apply(lambda x: f"{x:,.0f}"))
print(f"\nSmallest stocks (low volume):")
smallest = size_df.groupby('ticker')['size_proxy'].mean().nsmallest(10)
print(smallest.apply(lambda x: f"{x:,.0f}"))

Computing size factor...
Valid size observations: 216,694

Largest stocks (high volume):
ticker
SCOM    11,051,356
EQTY     1,996,915
KCB      1,732,197
KENO     1,030,634
COOP       890,976
MSC        883,201
ABSA       738,681
KEGN       608,506
KPLC       546,561
KNRE       511,337
Name: size_proxy, dtype: object

Smallest stocks (low volume):
ticker
LIMT      625
CITY      685
SMWF      998
KAPC    1,322
UTK     1,605
GLD     2,238
ORCH    2,447
BAUM    2,622
SGL     3,449
WTK     3,601
Name: size_proxy, dtype: object


In [7]:
# Size factor quintile analysis
merged_size = size_df.merge(
    returns[['ticker','date','adj_daily_return']],
    on=['ticker','date']
)

merged_size = merged_size.sort_values(['ticker','date'])
merged_size['fwd_return'] = merged_size.groupby('ticker')['adj_daily_return'].shift(-1)
merged_size = merged_size.dropna(subset=['fwd_return','size_proxy'])

# Log size — volume is highly skewed
merged_size['log_size'] = np.log1p(merged_size['size_proxy'])

quintile_rets = {q: [] for q in [1,2,3,4,5]}

for date, group in merged_size.groupby('date'):
    if len(group) < 15:
        continue
    try:
        group = group.copy()
        group['q'] = pd.qcut(
            group['log_size'], q=5,
            labels=[1,2,3,4,5], duplicates='drop'
        )
        for q in [1,2,3,4,5]:
            q_ret = group[group['q']==q]['fwd_return']
            if len(q_ret) >= 3:
                quintile_rets[q].append(q_ret.mean())
    except Exception:
        continue

print("=== SIZE FACTOR QUINTILES ===")
print("Q1 = smallest stocks, Q5 = largest stocks\n")

summary = {}
for q in [1,2,3,4,5]:
    s = pd.Series(quintile_rets[q]).dropna()
    if len(s) > 20:
        mean = s.mean()
        std = s.std()
        t = mean / (std / np.sqrt(len(s)))
        ann = mean * 252 * 100
        summary[q] = ann
        label = ' ← Small cap' if q==1 else ' ← Large cap' if q==5 else ''
        print(f"Q{q}: {ann:+6.1f}% ann  t={t:+.2f}{label}")

if 1 in summary and 5 in summary:
    spread = summary[1] - summary[5]
    print(f"\nSize premium (small minus large): {spread:+.1f}% ann")
    print(f"Direction: {'Small cap OUTPERFORMS' if spread > 0 else 'Large cap OUTPERFORMS'}")

=== SIZE FACTOR QUINTILES ===
Q1 = smallest stocks, Q5 = largest stocks

Q1:   +4.4% ann  t=+0.38 ← Small cap
Q2:   -6.2% ann  t=-0.49
Q3:   -1.7% ann  t=-0.12
Q4:   -9.2% ann  t=-0.57
Q5:   +5.7% ann  t=+0.35 ← Large cap

Size premium (small minus large): -1.4% ann
Direction: Large cap OUTPERFORMS


In [8]:
print("""
=== FINDING — SIZE FACTOR ===

Q1 (small cap): +4.4% ann, t=+0.38  NOT significant
Q5 (large cap): +5.7% ann, t=+0.35  NOT significant
Size premium:   -1.4% ann

CONCLUSION:
No size premium exists in NSE.
Neither small nor large caps systematically outperform.
All t-statistics below 1 — indistinguishable from noise.

LIKELY REASONS:
1. NSE small caps are simply illiquid — not tradeable
2. Transaction costs eliminate any theoretical small cap premium
3. Small cap NSE stocks often have fundamental problems
   (suspended trading, distressed companies)

Size factor REJECTED for Nairobi Alpha strategy.

CURRENT FACTOR RANKING:
  1. 20d mean reversion — CONFIRMED, t-stat 5.51
  2. Size proxy — REJECTED, t-stat < 1
  3. Long-horizon value proxies — REJECTED, not significant
""")


=== FINDING — SIZE FACTOR ===

Q1 (small cap): +4.4% ann, t=+0.38  NOT significant
Q5 (large cap): +5.7% ann, t=+0.35  NOT significant
Size premium:   -1.4% ann

CONCLUSION:
No size premium exists in NSE.
Neither small nor large caps systematically outperform.
All t-statistics below 1 — indistinguishable from noise.

LIKELY REASONS:
1. NSE small caps are simply illiquid — not tradeable
2. Transaction costs eliminate any theoretical small cap premium
3. Small cap NSE stocks often have fundamental problems
   (suspended trading, distressed companies)

Size factor REJECTED for Nairobi Alpha strategy.

CURRENT FACTOR RANKING:
  1. 20d mean reversion — CONFIRMED, t-stat 5.51
  2. Size proxy — REJECTED, t-stat < 1
  3. Long-horizon value proxies — REJECTED, not significant



In [3]:
# Factor: Illiquidity premium
# Amihud (2002) illiquidity measure:
# ILLIQ = |daily return| / daily volume (in KES)
# High ILLIQ = illiquid stock
# Hypothesis: illiquid stocks earn higher returns (illiquidity premium)
# Investors demand compensation for holding hard-to-sell stocks

print("Computing Amihud illiquidity factor...")

illiq_rows = []

for ticker, group in prices.groupby('ticker'):
    g = group.sort_values('date').copy()
    
    ret_data = returns[returns['ticker'] == ticker].set_index('date')
    g = g.set_index('date')
    g['abs_return'] = ret_data['adj_daily_return'].abs()
    g = g.reset_index()
    
    vol = pd.to_numeric(g['volume'], errors='coerce')
    price = pd.to_numeric(g['close'], errors='coerce')
    
    # Volume in KES = shares * price
    vol_kes = vol * price
    
    # Amihud measure
    illiq = np.where(
        vol_kes > 0,
        g['abs_return'] / vol_kes,
        np.nan
    )
    
    # Rolling 60-day average
    illiq_series = pd.Series(illiq, index=g.index)
    rolling_illiq = illiq_series.rolling(60, min_periods=30).mean()
    
    illiq_rows.append(pd.DataFrame({
        'ticker': ticker,
        'date': g['date'].values,
        'illiquidity': rolling_illiq.values,
    }))

illiq_df = pd.concat(illiq_rows, ignore_index=True)
illiq_df = illiq_df.dropna(subset=['illiquidity'])
illiq_df = illiq_df[illiq_df['illiquidity'] > 0]

print(f"Valid illiquidity observations: {len(illiq_df):,}")
print(f"\nMost illiquid (high Amihud):")
most_illiq = illiq_df.groupby('ticker')['illiquidity'].mean().nlargest(8)
print(most_illiq.apply(lambda x: f"{x:.2e}"))
print(f"\nMost liquid (low Amihud):")
most_liquid = illiq_df.groupby('ticker')['illiquidity'].mean().nsmallest(8)
print(most_liquid.apply(lambda x: f"{x:.2e}"))

Computing Amihud illiquidity factor...
Valid illiquidity observations: 180,838

Most illiquid (high Amihud):
ticker
NBV     2.22e-05
FTGH    1.55e-05
OCH     1.52e-05
DCON    1.31e-05
TCL     1.28e-05
BAUM    1.28e-05
UCHM    1.19e-05
XPRS    1.16e-05
Name: illiquidity, dtype: object

Most liquid (low Amihud):
ticker
SCOM    6.46e-10
KCB     1.30e-09
EQTY    2.27e-09
COOP    3.91e-09
BBK     4.99e-09
ABSA    6.37e-09
EABL    7.41e-09
KEGN    1.44e-08
Name: illiquidity, dtype: object


In [4]:
# Illiquidity factor quintile analysis
merged_illiq = illiq_df.merge(
    returns[['ticker','date','adj_daily_return']],
    on=['ticker','date']
)

merged_illiq = merged_illiq.sort_values(['ticker','date'])
merged_illiq['fwd_return'] = merged_illiq.groupby('ticker')['adj_daily_return'].shift(-1)
merged_illiq = merged_illiq.dropna(subset=['fwd_return','illiquidity'])

# Log illiquidity — highly skewed
merged_illiq['log_illiq'] = np.log(merged_illiq['illiquidity'])

quintile_rets = {q: [] for q in [1,2,3,4,5]}

for date, group in merged_illiq.groupby('date'):
    if len(group) < 15:
        continue
    try:
        group = group.copy()
        group['q'] = pd.qcut(
            group['log_illiq'], q=5,
            labels=[1,2,3,4,5], duplicates='drop'
        )
        for q in [1,2,3,4,5]:
            q_ret = group[group['q']==q]['fwd_return']
            if len(q_ret) >= 3:
                quintile_rets[q].append(q_ret.mean())
    except Exception:
        continue

print("=== ILLIQUIDITY FACTOR QUINTILES ===")
print("Q1 = most liquid, Q5 = most illiquid\n")

summary = {}
for q in [1,2,3,4,5]:
    s = pd.Series(quintile_rets[q]).dropna()
    if len(s) > 20:
        mean = s.mean()
        std = s.std()
        t = mean / (std / np.sqrt(len(s)))
        ann = mean * 252 * 100
        summary[q] = {'ann': ann, 't': t}
        label = ' ← Most liquid' if q==1 else ' ← Most illiquid' if q==5 else ''
        print(f"Q{q}: {ann:+6.1f}% ann  t={t:+.2f}{label}")

if 1 in summary and 5 in summary:
    premium = summary[5]['ann'] - summary[1]['ann']
    print(f"\nIlliquidity premium (Q5 minus Q1): {premium:+.1f}% ann")
    print(f"Direction: {'Illiquid OUTPERFORMS' if premium > 0 else 'Liquid OUTPERFORMS'}")

=== ILLIQUIDITY FACTOR QUINTILES ===
Q1 = most liquid, Q5 = most illiquid

Q1:   +1.9% ann  t=+0.12 ← Most liquid
Q2:   -7.6% ann  t=-0.50
Q3:   -3.1% ann  t=-0.23
Q4:   -5.1% ann  t=-0.34
Q5:   +2.3% ann  t=+0.16 ← Most illiquid

Illiquidity premium (Q5 minus Q1): +0.4% ann
Direction: Illiquid OUTPERFORMS


In [5]:
print("""
=== FINDING — ILLIQUIDITY FACTOR ===

Q1 (most liquid):   +1.9% ann, t=+0.12  NOT significant
Q5 (most illiquid): +2.3% ann, t=+0.16  NOT significant
Illiquidity premium: +0.4% ann

CONCLUSION:
No illiquidity premium exists in NSE.
T-statistics below 0.2 — pure noise.

LIKELY REASONS:
1. Illiquid NSE stocks are genuinely distressed or suspended
   — not just overlooked by investors
2. Transaction costs to buy illiquid stocks eliminate any premium
3. Exit risk — buying illiquid stocks means you may not be
   able to sell when you want to

Illiquidity factor REJECTED for Nairobi Alpha strategy.

=== FULL FACTOR SCORECARD ===

Factor              | Result    | T-stat | Decision
--------------------|-----------|--------|----------
20d mean reversion  | +83% ann  | +5.51  | CONFIRMED
6m value proxy      | -8% ann   | -0.53  | REJECTED
12m value proxy     | -13% ann  | -0.83  | REJECTED
Size (volume proxy) | -1% spread| <1.0   | REJECTED
Illiquidity (Amihud)| +0% spread| <0.2   | REJECTED

ONE FACTOR SURVIVES: 20d cross-sectional mean reversion.
This is the foundation of the Nairobi Alpha strategy.
""")


=== FINDING — ILLIQUIDITY FACTOR ===

Q1 (most liquid):   +1.9% ann, t=+0.12  NOT significant
Q5 (most illiquid): +2.3% ann, t=+0.16  NOT significant
Illiquidity premium: +0.4% ann

CONCLUSION:
No illiquidity premium exists in NSE.
T-statistics below 0.2 — pure noise.

LIKELY REASONS:
1. Illiquid NSE stocks are genuinely distressed or suspended
   — not just overlooked by investors
2. Transaction costs to buy illiquid stocks eliminate any premium
3. Exit risk — buying illiquid stocks means you may not be
   able to sell when you want to

Illiquidity factor REJECTED for Nairobi Alpha strategy.

=== FULL FACTOR SCORECARD ===

Factor              | Result    | T-stat | Decision
--------------------|-----------|--------|----------
20d mean reversion  | +83% ann  | +5.51  | CONFIRMED
6m value proxy      | -8% ann   | -0.53  | REJECTED
12m value proxy     | -13% ann  | -0.83  | REJECTED
Size (volume proxy) | -1% spread| <1.0   | REJECTED
Illiquidity (Amihud)| +0% spread| <0.2   | REJECTED

